# LLMOps Assignment 2 — API Cost, Latency, and Prompt Version Management
**Course:** MLOps | **Scenario:** EduAssist AI  
This notebook calls the configured LLM for the same five academic questions under three prompt versions. It records request latency and token-based cost, registers prompts in MLflow, logs aggregate metrics, checks JSON validity for v3, and assigns a `production` alias based on the measured results.

> **Important:** Running the notebook makes real API calls and may incur charges. Set the API key through an environment variable. Do not commit secrets. The notebook intentionally does not include fabricated results; generated CSVs reflect actual runs.

In [1]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from datetime import datetime

OLLAMA_HOST = "http://127.0.0.1:11434"
MODEL = "llama3.1:8b"

REQUEST_TIMEOUT = 600

print("Model:", MODEL)
print("Ollama URL:", OLLAMA_HOST)

Model: llama3.1:8b
Ollama URL: http://127.0.0.1:11434


In [2]:
response = requests.get(
    f"{OLLAMA_HOST}/api/tags",
    timeout=30
)

response.raise_for_status()

available_models = [
    model["name"]
    for model in response.json().get("models", [])
]

print("Available Ollama models:")
print(available_models)

if not any(
    name == MODEL or name.startswith(MODEL + ":")
    for name in available_models
):
    raise ValueError(
        f"{MODEL} is not installed. Run: ollama pull {MODEL}"
    )

print("\nOllama connection successful!")

Available Ollama models:
['llama3.1:8b']

Ollama connection successful!


In [3]:
questions = [
    "Explain Machine Learning in simple terms.",
    "What is the difference between supervised and unsupervised learning?",
    "What is overfitting and how can it be prevented?",
    "Why are APIs useful in machine learning applications?",
    "What is the purpose of CI/CD in MLOps?"
]

print("Total questions:", len(questions))

Total questions: 5


In [4]:
prompts = {
    "v1": """
You are an AI assistant.
Answer the following question clearly and briefly.

Question: {question}

Answer:
""",

    "v2": """
You are an expert machine learning instructor.
Explain the answer in simple language for a beginner.
Include a practical example where appropriate.
Keep the answer structured and concise.

Question: {question}

Answer:
""",

    "v3": """
You are a professional AI and MLOps instructor.
Answer the question accurately and concisely.

Return your response in valid JSON with this structure:
{
  "answer": "Detailed answer",
  "confidence": "High, Medium, or Low",
  "key_points": ["Point 1", "Point 2", "Point 3"]
}

Question: {question}
"""
}

for version, prompt in prompts.items():
    print(version, ":", prompt[:100], "...")

v1 : 
You are an AI assistant.
Answer the following question clearly and briefly.

Question: {question}

 ...
v2 : 
You are an expert machine learning instructor.
Explain the answer in simple language for a beginner ...
v3 : 
You are a professional AI and MLOps instructor.
Answer the question accurately and concisely.

Retu ...


In [6]:
def generate_response(prompt):

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2
        }
    }

    start_time = time.perf_counter()

    response = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json=payload,
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    result = response.json()

    latency = time.perf_counter() - start_time

    return {
        "response": result.get("response", ""),
        "latency_seconds": round(latency, 3),
        "prompt_tokens": result.get("prompt_eval_count", 0),
        "completion_tokens": result.get("eval_count", 0),
        "total_tokens": (
            result.get("prompt_eval_count", 0)
            + result.get("eval_count", 0)
        ),
        "total_duration": result.get("total_duration", 0) / 1e9,
        "load_duration": result.get("load_duration", 0) / 1e9,
        "prompt_eval_duration": (
            result.get("prompt_eval_duration", 0) / 1e9
        ),
        "eval_duration": result.get("eval_duration", 0) / 1e9
    }